import

In [ ]:
import os
import csv
import time
from typing import List, Any, Dict, Union
from dataclasses import dataclass

import numpy as np

import torch
from torch import (
    manual_seed,
    float64,
    Tensor,
    no_grad,
    argmax,
    max as torch_max
)
import torch.nn as nn
from torch.nn import (
    functional as F,
    Conv2d,
    ReLU,
    Linear,
    Flatten,
    Softmax,
    Sequential,
    Module,
    DataParallel,
    MSELoss,
    CrossEntropyLoss
)
from torch.optim.adam import Adam
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder
from torchvision.transforms import Compose, Resize, Grayscale, ToTensor

import pennylane as qml
from pennylane.operation import AnyWires, Operation
from pennylane.ops import RX, RY, RZ
from pennylane.ops.channel import DepolarizingChannel
from pennylane.wires import WiresLike
from pennylane.typing import TensorLike
from pennylane.measurements import ProbabilityMP
from pennylane.qnn import TorchLayer
from pennylane.ops.qubit.parametric_ops_multi_qubit import IsingZZ

import hydra
from omegaconf import DictConfig

import matplotlib.pyplot as plt

manual_seed(42)

dataset.py

In [ ]:
def clear_folder(folder_path: str) -> None:
    """Remove all hidden files and/or hidden subfolder
    from a given folder.

    Parameters
    ----------
    folder_path : str
        The name of the folder to be cleared.

    Returns
    -------
    None
        This function simply modifies the input folder
    """
    content = os.listdir(folder_path)
    hidden_files = [f for f in content if f.startswith(".")]
    for hidden_file in hidden_files:
        hidden_file_path = os.path.join(folder_path, hidden_file)
        if os.path.isdir(hidden_file_path):
            os.rmdir(hidden_file_path)  # Remove directory if it's hidden
        else:
            os.remove(hidden_file_path)  # Remove file if it's hidden


def num_classes(dataset_folder_path: str) -> int:
    """Return the number of classes the data of a given dataset are
    categorized in by assuming that all the data are contained within
    a folder with the following structure:
    dataset_folder
        ├── Training
            ├── Class 1
            ├── Class 2
            ├── Class 3
            ...
        ├── Test
            ├── Class 1
            ├── Class 2
            ├── Class 3
            ...
        ├── Validation
            ├── Class 1
            ├── Class 2
            ├── Class 3
            ...
    To determine it, this function counts the number of subsubfolders
    contained within the Training subfolder after removing all hidden
    files and/or hidden subsubfolders.

    Parameters
    ----------
    dataset : str
        The name of the dataset folder.

    Returns
    -------
    int
        The number of classes the data are categorized in.
    """
    # Define the directory for training set
    train_dir = os.path.join(dataset_folder_path, "Training")

    # Remove hidden folders in the training set directory
    clear_folder(folder_path=train_dir)

    # determine the number of classes by counting the number of folders
    num_classes = len(
        [
            name
            for name in os.listdir(train_dir)
            if os.path.isdir(os.path.join(train_dir, name))
        ]
    )
    return num_classes


def clear_dataset(dataset_folder_path: str) -> None:
    """Given a dataset folder with the following structure:
    dataset_folder_path
        ├── Training
            ├── Class 1
            ├── Class 2
            ├── Class 3
            ...
        ├── Test
            ├── Class 1
            ├── Class 2
            ├── Class 3
            ...
        ├── Validation
            ├── Class 1
            ├── Class 2
            ├── Class 3
            ...
    remove all hidden files and/or folders from each folder.

    Parameters
    ----------
    dataset : str
        The name of the dataset folder.

    Returns
    -------
    None
        This function simply modifies the input folder.
    """
    clear_folder(dataset_folder_path)

    train_dir = os.path.join(dataset_folder_path, "Training")
    clear_folder(train_dir)
    val_dir = os.path.join(dataset_folder_path, "Validation")
    clear_folder(val_dir)
    test_dir = os.path.join(dataset_folder_path, "Test")
    clear_folder(test_dir)

    dirs = [
        os.path.join(train_dir, "L"),
        os.path.join(train_dir, "O"),
        os.path.join(train_dir, "S"),
        os.path.join(train_dir, "T"),
        os.path.join(val_dir, "L"),
        os.path.join(val_dir, "O"),
        os.path.join(val_dir, "S"),
        os.path.join(val_dir, "T"),
        os.path.join(test_dir, "L"),
        os.path.join(test_dir, "O"),
        os.path.join(test_dir, "S"),
        os.path.join(test_dir, "T"),
    ]

    for dir in dirs:
        clear_folder(dir)


def load_dataset(
    dataset_folder_path: str,
    batch_size: int,
    drop_last: bool = True,
) -> tuple:
    """Given a dataset folder with the following structure:
    dataset_folder
        ├── Training
            ├── Class 1
            ├── Class 2
            ├── Class 3
            ...
        ├── Test
            ├── Class 1
            ├── Class 2
            ├── Class 3
            ...
        ├── Validation
            ├── Class 1
            ├── Class 2
            ├── Class 3
            ...
    return the training, validation and test sets. Differently from the last
    two, the first is divided into batches.

    Parameters
    ----------
    dataset_folder_path : str
        The path of the folder containing the dataset.
    batch_size : int
        The size of the batch into which the training dataset is
        divided during the training phase.
    drop_last : bool
        Discard the last of the training dataset batch if incomplete

    Returns
    -------
    tuple
        A tuple containing the training, validation and test sets.
    """

    # Define the directories for train, validation and test
    clear_dataset(dataset_folder_path)

    train_dir = os.path.join(dataset_folder_path, "Training")
    validation_dir = os.path.join(dataset_folder_path, "Validation")
    test_dir = os.path.join(dataset_folder_path, "Test")

    # Load datasets
    n_classes = num_classes(dataset_folder_path=dataset_folder_path)

    # Pre-processing operations for data and labels
    transform = Compose([Resize(3), Grayscale(num_output_channels=1), ToTensor()])
    target_transform = Compose(
        [
            lambda x: torch.tensor(x),
            lambda x: torch.eye(n=n_classes)[x].to(torch.float64),  # ohe
        ]
    )

    train_dataset = ImageFolder(
        root=train_dir, transform=transform, target_transform=target_transform
    )
    validation_dataset = ImageFolder(
        root=validation_dir, transform=transform, target_transform=target_transform
    )
    test_dataset = ImageFolder(
        root=test_dir, transform=transform, target_transform=target_transform
    )

#    # Limit the datasets to the first 10 images
#    train_dataset = Subset(train_dataset, range(min(10, len(train_dataset))))
#    validation_dataset = Subset(validation_dataset, range(min(10, len(validation_dataset))))
#    test_dataset = Subset(test_dataset, range(min(10, len(test_dataset))))

    # Create data loaders for train, validation, and test datasets
    train_loader = DataLoader(
        dataset=train_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=drop_last
    )
    validation_loader = DataLoader(
        dataset=validation_dataset,
        batch_size=len(validation_dataset),
        shuffle=False
    )
    test_loader = DataLoader(
        dataset=test_dataset,
        batch_size=len(test_dataset),
        shuffle=False
    )
    return train_loader, validation_loader, test_loader

operations.py

In [ ]:
ROT = {"X": RX, "Y": RY, "Z": RZ}

class AngleEmbedding(Operation):
    r"""
    Encodes :math:`N` features into the rotation angles of :math:`n` qubits, where :math:`N \leq n`.

    The rotations can be chosen as either :class:`~pennylane.ops.RX`, :class:`~pennylane.ops.RY`
    or :class:`~pennylane.ops.RZ` gates, as defined by the ``rotation`` parameter:

    * ``rotation='X'`` uses the features as angles of RX rotations

    * ``rotation='Y'`` uses the features as angles of RY rotations

    * ``rotation='Z'`` uses the features as angles of RZ rotations

    The length of ``features`` has to be smaller or equal to the number of qubits. If there are fewer entries in
    ``features`` than rotations, the circuit does not apply the remaining rotation gates.

    Args:
        features (tensor_like): input tensor of shape ``(N,)``, where N is the number of input features to embed,
            with :math:`N\leq n`
        wires (Any or Iterable[Any]): wires that the template acts on
        rotation (str): type of rotations used
        id (str): custom label given to an operator instance,
            can be useful for some applications where the instance has to be identified.

    Example:

        Angle embedding encodes the features by using the specified rotation operation.

        .. code-block:: python

            dev = qml.device('default.qubit', wires=3)

            @qml.qnode(dev)
            def circuit(feature_vector):
                qml.AngleEmbedding(features=feature_vector, wires=range(3), rotation='Z')
                qml.Hadamard(0)
                return qml.probs(wires=range(3))

            X = [1,2,3]

        Here, we have also used rotation angles :class:`RZ`. If not specified, :class:`RX` is used as default.
        The resulting circuit is:

        >>> print(qml.draw(circuit, level="device")(X))
        0: ──RZ(1.00)──H─┤ ╭Probs
        1: ──RZ(2.00)────┤ ├Probs
        2: ──RZ(3.00)────┤ ╰Probs

    """

    num_wires = AnyWires
    grad_method = None # type: ignore

    def _flatten(self):
        hyperparameters = (("rotation", self._rotation),)
        return self.data, (self.wires, hyperparameters)

    def __repr__(self):
        return f"AngleEmbedding({self.data[0]}, wires={self.wires.tolist()}, rotation={self._rotation})"

    def __init__(self,
                 features,
                 wires,
                 noise : str | None = None,
                 noise_prob : float | None = None,
                 rotation="X",
                 id=None):
        if rotation not in ROT:
            raise ValueError(f"Rotation option {rotation} not recognized.")

        shape = qml.math.shape(features)[-1:]
        n_features = shape[0]
        if n_features > len(wires):
            raise ValueError(
                f"Features must be of length {len(wires)} or less; got length {n_features}."
            )

        self._rotation = rotation
        self._hyperparameters = {
            "rotation": ROT[rotation],
            "noise": noise,
            "noise_prob": noise_prob
        }

        wires = wires[:n_features]
        super().__init__(features, wires=wires, id=id)

    @property
    def num_params(self):
        return 1

    @property
    def ndim_params(self):
        return (1,)

    @staticmethod
    def compute_decomposition(
        features,
        wires,
        rotation,
        noise,
        noise_prob
    ):  # pylint: disable=arguments-differ
        r"""Representation of the operator as a product of other operators.

        .. math:: O = O_1 O_2 \dots O_n.



        .. seealso:: :meth:`~.AngleEmbedding.decomposition`.

        Args:
            features (tensor_like): input tensor of dimension ``(len(wires),)``
            wires (Any or Iterable[Any]): wires that the operator acts on
            rotation (.Operator): rotation gate class

        Returns:
            list[.Operator]: decomposition of the operator

        **Example**

        >>> features = torch.tensor([1., 2.])
        >>> qml.AngleEmbedding.compute_decomposition(features, wires=["a", "b"], rotation=qml.RX)
        [RX(tensor(1.), wires=['a']),
         RX(tensor(2.), wires=['b'])]
        """
        batched = qml.math.ndim(features) > 1
        # We will iterate over the first axis of `features` together with iterating over the wires.
        # If the leading dimension is a batch dimension, exchange the wire and batching axes.
        features = qml.math.T(features) if batched else features
        
        decomposition = []
        for i in range(len(wires)):
            decomposition.append(rotation(features[i], wires=wires[i]))
            if noise == 'depolarizing' and noise_prob is not None and noise_prob > 0:
                decomposition.append(DepolarizingChannel(p=noise_prob, wires=wires[i]))

        return decomposition


class RealAmplitudes(Operation):
    r"""Layers consisting of single qubit rotations and entanglers, inspired by the circuit-centric classifier design
    `arXiv:1804.00633 <https://arxiv.org/abs/1804.00633>`_.

    The argument ``weights`` contains the weights for each layer. The number of layers :math:`L` is therefore derived
    from the first dimension of ``weights``.

    The 2-qubit gates, whose type is specified by the ``imprimitive`` argument,
    act chronologically on the :math:`M` wires, :math:`i = 1,...,M`. The second qubit of each gate is given by
    :math:`(i+r)\mod M`, where :math:`r` is a  hyperparameter called the *range*, and :math:`0 < r < M`.
    If applied to one qubit only, this template will use no imprimitive gates.

    This is an example of two 4-qubit strongly entangling layers (ranges :math:`r=1` and :math:`r=2`, respectively) with
    rotations :math:`RY` and CNOTs as imprimitives:

    .. figure:: ../../_static/layer_sec_ry.png
        :align: center
        :width: 60%
        :target: javascript:void(0);

    .. note::
        The two-qubit gate used as the imprimitive or entangler must not depend on parameters.

    Args:

        weights (tensor_like): weight tensor of shape ``(L, M)``
        wires (Iterable): wires that the template acts on
        ranges (Sequence[int]): sequence determining the range hyperparameter for each subsequent layer; if ``None``
                                using :math:`r=l \mod M` for the :math:`l` th layer and :math:`M` wires.
        imprimitive (type of pennylane.ops.Operation): two-qubit gate to use, defaults to :class:`~pennylane.ops.CNOT`

    Example:

        There are multiple arguments that the user can use to customize the layer.

        The required arguments are ``weights`` and ``wires``.

        .. code-block:: python

            dev = qml.device('default.qubit', wires=4)

            @qml.qnode(dev)
            def circuit(parameters):
                qml.StronglyEntanglingLayers(weights=parameters, wires=range(4))
                return qml.expval(qml.Z(0))

            shape = qml.StronglyEntanglingLayers.shape(n_layers=2, n_wires=4)
            weights = np.random.random(size=shape)

        The shape of the ``weights`` argument decides the number of layers.

        The resulting circuit is:

        >>> print(qml.draw(circuit, level="device")(weights))
        0: ──RY(0.68)─╭●───────╭X──RY(0.94)─╭●────╭X────┤  <Z>
        1: ──RY(0.91)─╰X─╭●────│───RY(0.50)─│──╭●─│──╭X─┤
        2: ──RY(0.91)────╰X─╭●─│───RY(0.14)─╰X─│──╰●─│──┤
        3: ──RY(0.46)───────╰X─╰●──RY(0.87)────╰X────╰●─┤

        The default two-qubit gate used is :class:`~pennylane.ops.CNOT`. This can be changed by using the ``imprimitive`` argument.

        The ``ranges`` argument takes an integer sequence where each element
        determines the range hyperparameter for each layer. This range hyperparameter
        is the difference of the wire indices representing the two qubits the
        ``imprimitive`` gate acts on. For example, for ``range=[2,3]`` the
        first layer will have a range parameter of ``2`` and the second layer will
        have a range parameter of ``3``.
        Assuming ``wires=[0, 1, 2, 3]`` and a range parameter of ``2``, there will be
        an imprimitive gate acting on:

        * qubits ``(0, 2)``;
        * qubits ``(1, 3)``;
        * qubits ``(2, 0)``;
        * qubits ``(3, 1)``.

        .. code-block:: python

            dev = qml.device('default.qubit', wires=4)

            @qml.qnode(dev)
            def circuit(parameters):
                qml.StronglyEntanglingLayers(weights=parameters, wires=range(4), ranges=[2, 3], imprimitive=qml.ops.CZ)
                return qml.expval(qml.Z(0))

            shape = qml.StronglyEntanglingLayers.shape(n_layers=2, n_wires=4)
            weights = np.random.random(size=shape)

        The resulting circuit is:

        >>> print(qml.draw(circuit, level="device")(weights))
        0: ──RY(0.99)─╭●────╭Z──RY(0.02)──────────────────────╭●─╭Z───────┤  <Z>
        1: ──RY(0.55)─│──╭●─│──╭Z────────────────────RY(0.15)─│──╰●─╭Z────┤
        2: ──RY(0.79)─╰Z─│──╰●─│─────────────────────RY(0.73)─│─────╰●─╭Z─┤
        3: ──RY(0.30)────╰Z────╰●────────────────────RY(0.57)─╰Z───────╰●─┤

    .. details::
        :title: Usage Details

        **Parameter shape**

        The expected shape for the weight tensor can be computed with the static method
        :meth:`~.qml.StronglyEntanglingLayers.shape` and used when creating randomly
        initialised weight tensors:

        .. code-block:: python

            shape = qml.StronglyEntanglingLayers.shape(n_layers=2, n_wires=2)
            weights = np.random.random(size=shape)

    """

    num_wires = AnyWires
    grad_method = None # type: ignore

    def __init__(
            self,
            weights,
            wires,
            noise: str | None=None,
            noise_prob: float | None=None,
            ranges=None,
            imprimitive=None,
            id=None
        ):
        
        shape = qml.math.shape(weights)[-2:]
        self.noise = noise
        self.noise_prob = noise_prob

        if shape[1] != len(wires):
            raise ValueError(
                f"Weights tensor must have second dimension of length {len(wires)}; got {shape[1]}"
            )

        if len(shape) != 2:
            raise ValueError(
                f"Weights tensor must have shape (n_layers, n_wires); got {shape}"
            )

        if ranges is None:
            if len(wires) > 1:
                # tile ranges with iterations of range(1, n_wires)
                ranges = tuple((l % (len(wires) - 1)) + 1 for l in range(shape[0]))
            else:
                ranges = (0,) * shape[0]
        else:
            ranges = tuple(ranges)
            if len(ranges) != shape[0]:
                raise ValueError(f"Range sequence must be of length {shape[0]}; got {len(ranges)}")
            for r in ranges:
                if r % len(wires) == 0:
                    raise ValueError(
                        f"Ranges must not be zero nor divisible by the number of wires; got {r}"
                    )

        self._hyperparameters = {
            "ranges": ranges,
            "imprimitive": imprimitive or qml.CNOT,
            "noise": noise,
            "noise_prob": noise_prob
        }

        super().__init__(weights, wires=wires, id=id)

    @property
    def num_params(self):
        return 1

    @staticmethod
    def compute_decomposition(
        weights: TensorLike, wires, ranges, imprimitive, noise, noise_prob
    ):  # pylint: disable=arguments-differ
        r"""Representation of the operator as a product of other operators.

        .. math:: O = O_1 O_2 \dots O_n.



        .. seealso:: :meth:`~.StronglyEntanglingLayers.decomposition`.

        Args:
            weights (tensor_like): weight tensor
            wires (Any or Iterable[Any]): wires that the operator acts on
            ranges (Sequence[int]): sequence determining the range hyperparameter for each subsequent layer
            imprimitive (pennylane.ops.Operation): two-qubit gate to use

        Returns:
            list[.Operator]: decomposition of the operator

        **Example**

        >>> weights = torch.tensor([[-0.2, 0.1], [1.2, -2.]])
        >>> qml.StronglyEntanglingLayers.compute_decomposition(weights, wires=["a", "b"], ranges=[2], imprimitive=qml.CNOT)
        [RY(tensor(-0.2000), wires=['a']),
        RY(tensor(0.1000), wires=['b']),
        CNOT(wires=['a', 'a']),
        RY(tensor(1.2000), wires=['a']),
        RY(tensor(-2.), wires=['b']),
        CNOT(wires=['b', 'b'])]
        """
        n_layers = qml.math.shape(weights)[-2]
        wires = qml.wires.Wires(wires)
        op_list = []

        for l in range(n_layers):
            for i in range(len(wires)):  # pylint: disable=consider-using-enumerate
                op_list.append(
                    qml.RY(
                        weights[..., l, i],
                        wires=wires[i],
                    )
                )
                if noise == "depolarizing" and noise_prob is not None and noise_prob > 0:
                    op_list.append(DepolarizingChannel(p=noise_prob, wires=wires[i]))

            if len(wires) > 1:
                for i in range(len(wires)):
                    act_on = wires.subset([i, i + ranges[l]], periodic_boundary=True)
                    op_list.append(imprimitive(wires=act_on))
                    if noise == "depolarizing" and noise_prob is not None and noise_prob > 0:
                        op_list.append(DepolarizingChannel(p=noise_prob, wires=i))

        return op_list

    @staticmethod
    def shape(n_layers, n_wires):
        r"""Returns the expected shape of the weights tensor.

        Args:
            n_layers (int): number of layers
            n_wires (int): number of wires

        Returns:
            tuple[int]: shape
        """

        return n_layers, n_wires

    # pylint:disable = no-value-for-parameter
    @staticmethod
    def compute_qfunc_decomposition(
        weights, *wires, ranges, imprimitive, noise, noise_prob
    ):  # pylint: disable=arguments-differ
        wires = qml.math.array(wires, like="jax")
        ranges = qml.math.array(ranges, like="jax")

        n_wires = len(wires)
        n_layers = weights.shape[0]

        @qml.for_loop(n_layers)
        def layers(l):
            @qml.for_loop(n_wires)
            def rot_loop(i):
                qml.RY(
                    weights[l, i],
                    wires=wires[i],
                )
                if noise == "depolarizing" and noise_prob is not None and noise_prob > 0:
                    DepolarizingChannel(p=noise_prob, wires=wires[i])

            def imprim_true():
                @qml.for_loop(n_wires)
                def imprimitive_loop(i):
                    act_on = qml.math.array([i, i + ranges[l]], like="jax") % n_wires
                    imprimitive(wires=wires[act_on])
                    if noise == "depolarizing" and noise_prob is not None and noise_prob > 0:
                        DepolarizingChannel(p=noise_prob, wires=wires[act_on])

                imprimitive_loop()

            def imprim_false():
                pass

            rot_loop()
            qml.cond(n_wires > 1, imprim_true, imprim_false)()

        layers()

quanvolution.py

In [ ]:
def z_feature_map(
        input_features: Tensor,
        reps: int,
        noise: str | None = None,
        noise_prob: float | None = None,
    ) -> None:
    """Z feature map for the VQC."""
    if len(input_features) < 1:
        raise ValueError("Number of features must be at least 1.")
    if reps < 1:
        raise ValueError("Feature map repetitions must be at least 1.")

    for r in range(reps):
        for i in range(len(input_features)):
            qml.Hadamard(wires=i)
        AngleEmbedding(
            features=[2*feature for feature in input_features],
            wires=range(len(input_features)),
            rotation='Y',
            noise=noise,
            noise_prob=noise_prob
        )

def zz_feature_map(
        input_features: Tensor,
        reps: int,
        noise: str | None = None,
        noise_prob : float | None = None
    ) -> None:
    """ZZ feature map for the VQC."""

    if len(input_features) < 1:
        raise ValueError("Number of features must be at least 1.")
    if reps < 1:
        raise ValueError("Feature map repetitions must be at least 1.")

    for r in range(reps):
        for i in range(len(input_features)):
            qml.Hadamard(wires=i)
        AngleEmbedding(
            features=[2 * feature for feature in input_features],
            wires=range(len(input_features)),
            rotation='Y',
            noise=noise,
            noise_prob=noise_prob
        )
        for i in range(len(input_features) - 1):
            phi_val : TensorLike = 2 * (np.pi - input_features[i]) * (np.pi - input_features[i+1]) # type: ignore
            IsingZZ(wires=[i + 1,i], phi=phi_val)
        if noise == 'depolarizing':    
            for i in range(len(input_features)):
                DepolarizingChannel(p=noise_prob, wires=i)

def real_amplitudes_ansatz(
        num_qubits: int,
        reps: int,
        params: Tensor,
        noise: str | None,
        noise_prob: float | None
    ) -> None:
    """Ansatz for the VQC."""

    if reps < 1:
        raise ValueError("Feature map repetitions must be at least 1.")
    for r in range(reps):
        RealAmplitudes(
            weights=params,
            wires=range(num_qubits),
            noise=noise,
            noise_prob=noise_prob
        )

class Quanvolution(nn.Module):
    """Quanvolutional layer for quantum convolutional neural networks."""

    def __init__(
        self,
        device: qml.devices,
        noise: str | None,
        noise_prob: float | None,
        feature_map: str,
        ansatz: str,
        feature_map_reps: int,
        ansatz_reps: int,
        qfilter_size: int,
        show_circuit: bool=False
    ) -> None:
        
        super(Quanvolution, self).__init__()
        self.device = device
        self.feature_map = feature_map
        self.ansatz = ansatz
        self.feature_map_reps = feature_map_reps
        self.ansatz_reps = ansatz_reps
        self.show_circuit = show_circuit
        self.num_qubits : int = int(qfilter_size * qfilter_size)
        self.output_channels = int(2 ** self.num_qubits)
        self.qfilter_size = qfilter_size

        # Define the quantum filter
        @qml.qnode(
            device=device,
            interface='torch',
            diff_method='parameter-shift'
        )
        def qnode(
            inputs: Tensor,
            params: Tensor
        ) -> ProbabilityMP:
            """Quantum circuit for the VQC."""
            if feature_map not in ['z', 'zz']:
                raise ValueError("Feature map must be 'z' or 'zz'.")
            if ansatz not in ['real_amplitudes']:
                raise ValueError("Ansatz must be 'real_amplitudes'.")

            num_qubits: int = int(qfilter_size * qfilter_size)

            if feature_map == 'z':
                z_feature_map(input_features=inputs, reps=feature_map_reps)
            elif feature_map == 'zz':
                zz_feature_map(input_features=inputs, reps=feature_map_reps)

            if ansatz == 'real_amplitudes':
                real_amplitudes_ansatz(
                    num_qubits=num_qubits,
                    reps=ansatz_reps,
                    params=params,
                    noise=noise,
                    noise_prob=noise_prob
                )
            # if self.show_circuit:
            #     print(qml.draw(qnode)(inputs, params))
            
            return qml.probs(wires=range(num_qubits))

        # Calculate the shape of the parameters
        weight_shape = {"params": ((ansatz_reps + 1),(qfilter_size ** 2))}

        self.qfilter = TorchLayer(qnode=qnode, weight_shapes=weight_shape) # type: ignore

    def forward(self, data_loader: Tensor) -> Tensor:
        # Unfold the input tensor to prepare it for quantum processing
        # print('input data shape: ', data_loader.shape)
        input_unfolded: Tensor = F.unfold(
            input=data_loader,
            kernel_size=int(self.qfilter_size),
        ).transpose(1, 2)
        # print('input unfolded shape: ', input_unfolded.shape)

        # Reshape the unfolded input to extract sliding blocks
        input_unfolded_reshaped: Tensor = input_unfolded.reshape(
            input_unfolded.size(0) * input_unfolded.size(1), -1
        )
        # print('input unfolded reshaped shape: ', input_unfolded_reshaped.shape)

        # Apply the quantum filter every sliding block
        output_unfolded : Tensor = torch.zeros(size=(input_unfolded_reshaped.size(0), self.output_channels))
        for i in range(input_unfolded_reshaped.size(0)):
            sliding_block : Tensor = input_unfolded_reshaped[i].squeeze()
            output_unfolded[i] = self.qfilter(sliding_block)
        # print('output unfolded shape: ', output_unfolded.shape)

        # Reshape the output to match the original unfolded input shape
        output_unfolded_reshaped: Tensor = output_unfolded.view(
            input_unfolded.size(0), input_unfolded.size(1), -1
        )
        # print('output unfolded reshaped shape: ', output_unfolded_reshaped.shape)

        # Transpose the reshaped output to prepare for refolding
        output_unfolded_reshaped: Tensor = output_unfolded_reshaped.transpose(1, 2)
        # print('output unfolded reshaped transposed shape: ', output_unfolded_reshaped.shape)

        # Refold the output tensor to its original spatial dimensions
        output_refolded: Tensor = output_unfolded_reshaped.view(
            input_unfolded.size(0),
            output_unfolded.size(1),
            int(output_unfolded_reshaped.size(2) ** 0.5),
            int(output_unfolded_reshaped.size(2) ** 0.5),
        )
        # print('output refolded shape: ', output_refolded.shape)

        return output_refolded

net.py

In [ ]:
class ClassicNet(Module):
    """Convolutional Neural Network composed of a single convolutional layer,
    followed by a single fully connected layer and a softmax.
    """

    def __init__(
        self,
        kernel_size: int,
        convolution_output_channels: int,
        classifier_input_features: int,
        classifier_output_features: int
    ):
        super(ClassicNet, self).__init__()

        self.kernel_size = kernel_size
        self.convolution_output_channels = convolution_output_channels
        self.classifier_input_features = classifier_input_features
        self.classifier_output_features = classifier_output_features

        self.convolution = Conv2d(
            in_channels=1,
            out_channels=self.convolution_output_channels,
            kernel_size=kernel_size
        )

        self.net = Sequential(
            self.convolution,
            ReLU(),
            Flatten(),
            Linear(
                in_features=classifier_input_features,
                out_features=classifier_output_features
            ),
            Softmax(dim=1)
        )

        self.prob = None
    def forward(self, x: Tensor) -> Tensor:
        # print('Model parameters:', self.convolution.state_dict())
        return self.net(x)


class HybridNet(Module):
    """Convolutional Neural Network composed of a single convolutional layer,
    followed by a single fully connected layer and a softmax.
    """

    def __init__(
        self,
        device: qml.devices,
        noise: str | None,
        noise_prob: float | None,
        feature_map: str,
        ansatz: str,
        feature_map_reps: int,
        ansatz_reps: int,
        qfilter_size: int,
        classifier_input_features: int,
        classifier_output_features: int,
        show_circuit: bool = False,
    ):
        super(HybridNet, self).__init__()

        self.prob = noise_prob
        
        self.quanvolution = Quanvolution(
            device=device,
            noise=noise,
            noise_prob=noise_prob,
            feature_map=feature_map,
            ansatz=ansatz,
            feature_map_reps=feature_map_reps,
            ansatz_reps=ansatz_reps,
            qfilter_size=qfilter_size,
            show_circuit=show_circuit
     )

        self.net = Sequential(
            self.quanvolution,
            Flatten(),
            Linear(
                in_features=classifier_input_features,
                out_features=classifier_output_features,
            ),
            Softmax(dim=1)
        )

    def forward(self, x: Tensor) -> Tensor:
        # print('Model parameters:', self.quanvolution.state_dict())
        return self.net(x)


def flatten_dimension(
    train_loader: DataLoader,
    kernel_size: int,
    convolution_output_channels: int,
) -> int:
    """Determine the number of neurons obtained by flattening the output
    images of the convolutional layer.
    """

    # Determine the width of the images
    images, _ = next(iter(train_loader))
    in_width: int = images.shape[3]

    # Determine the width of the kernel
    k_width: int = int(kernel_size)
#    print('Kernel size:', k_width)

    # Determine the width of the output images
    out_width: int = int(in_width - k_width + 1)

    # Determine the number of pixels in each output image
    out_pixels: int = int(out_width * out_width)
#    print('Output image size:', out_pixels)

    # Determine the total number of pixel
    flatten_size: int = out_pixels * convolution_output_channels
#    print('Flatten size:', flatten_size)

    return flatten_size


def create_cnn(
    train_loader: DataLoader,
    dataset_folder_path: str,
    kernel_size: int,
    device: qml.devices,
    noise: str | None,
    noise_prob: float | None,
    feature_map: str,
    ansatz: str,
    feature_map_reps: int,
    ansatz_reps: int,
    classes: int,
    show_circuit: bool = False,
) -> Union[HybridNet, ClassicNet]:
    """Create either a classical or a hybrid convolutional neural network
    composed of a single convolutional layer, a single dense layer.
    """

    convolution_output_channels: int = int(2 ** (kernel_size * kernel_size))

    # Determine the number of input features of the classifier
    classifier_input_features: int = flatten_dimension(
        train_loader=train_loader,
        kernel_size=kernel_size,
        convolution_output_channels=convolution_output_channels,
    )

    # Determine the number of classes
    classifier_output_features: int = num_classes(
        dataset_folder_path=dataset_folder_path
    )

    # Create either the classical or the hybrid cnn
    model: Module
    if noise_prob is None:
        model = ClassicNet(
            kernel_size=kernel_size,
            convolution_output_channels=convolution_output_channels,
            classifier_input_features=classifier_input_features,
            classifier_output_features=classifier_output_features,
        )
    else:
        model = HybridNet(
        device = device,
        noise = noise,
        noise_prob = noise_prob,
        feature_map = feature_map,
        ansatz = ansatz,
        feature_map_reps = feature_map_reps,
        ansatz_reps=ansatz_reps,
        qfilter_size=kernel_size,
        classifier_input_features = classifier_input_features,
        classifier_output_features = classes,
        show_circuit = show_circuit
    )

    return model

training.py

In [ ]:
@dataclass
class TrainingResult:
    avg_epoch_train_costs: List[Tensor]
    avg_epoch_train_accuracies: List[Tensor]
    avg_epoch_test_costs: List[Tensor]
    avg_epoch_test_accuracies: List[Tensor]
    models: List[Dict[str, Any]]
    plot_path: str

class Trainer:
    """Class to train and validate a module.

    Attributes
    ----------
    model : Union[ClassicNet,HybridNet]
        The model to be trained.
    train_loader : DataLoader
        The data loader of the training set.
    test_loader : DataLoader
        The data loader of the test set.
    loss_fn : Union[MSELoss, CrossEntropyLoss]
        The loss function used to optimize the parameters.
    epochs : int
        The number of epochs of the training.
    learning_rate : float
        The learning rate used by the optimizer.

    Methods
    -------
    train_and_validate
        Performs both training and test on the dataset and saves the
        metrics along the way.
    """

    def __init__(
        self,
        model: Union[ClassicNet, HybridNet],
        train_loader: DataLoader,
        test_loader: DataLoader,
        loss_fn: Union[MSELoss, CrossEntropyLoss],
        epochs: int,
        learning_rate: float,
    ):
        self.model = DataParallel(model)
        self.epochs = epochs
        self.train_loader = train_loader
        self.test_loader = test_loader
        self.loss_fn = loss_fn
        self.learning_rate = learning_rate

        path : str
        if model.prob is None:
            path = 'classical'
        else :
            path = str(model.prob) + '%'

        # Create the output folder if it doesn't exist
        if not os.path.exists('results'):
            os.makedirs('results')
        if not os.path.exists('plots'):
            os.makedirs('plots')

        self.csv_path = os.path.join('results', path + '.csv')
        self.plot_path = os.path.join('plots', path + '.pdf')

    def train_and_validate(self) -> Union[TrainingResult, None]:
        model = self.model
        # Initialize the results object
        results = TrainingResult([], [], [], [], [], self.plot_path)

        with open(self.csv_path, "w", newline="") as csvfile:
            # Create a csv writer object
            csvwriter = csv.writer(csvfile)

            # Write the header
            csvwriter.writerow(
                [
                    "Epoch",
                    "Train Loss",
                    "Train Accuracy",
                    "Test Loss",
                    "Test Accuracy",
                ]
            )

            for epoch in range(self.epochs):
                start_epoch_time = time.time()
                epoch_train_costs: List[Tensor] = []
                epoch_train_accuracies: List[Tensor] = []
                epoch_test_costs: List[Tensor] = []
                epoch_test_accuracies: List[Tensor] = []

                # Initialize the optimizer
                optimizer = Adam(params=model.parameters(), lr=self.learning_rate)

                # Train the model
                model.train()

                for batch_index, (inputs, labels) in enumerate(self.train_loader):
                    # print('EPOCH: ', epoch + 1)
                    # print('TRAIN BATCH: ', batch_index + 1)
                    # Start recording time
                    start_train_time = time.time()

                    optimizer.zero_grad()

                    output = model(inputs)

                    # Compute accuracy
                    _, predicted_labels = torch_max(output, 1)
                    true_labels = argmax(labels, dim=1)
                    correct_train_predictions: Tensor = (
                        predicted_labels == true_labels
                    ).sum()
                    train_accuracy: Tensor = correct_train_predictions / inputs.size(0)

                    # Optimize parameters
                    train_cost_fn: Tensor = self.loss_fn(output, labels.float())
                    train_cost_fn.backward()
                    optimizer.step()

                    # Add metrics to lists
                    epoch_train_costs.append(train_cost_fn)
                    epoch_train_accuracies.append(train_accuracy)

                    # End recording time and compute total time
                    end_train_time = time.time()
                    train_time = end_train_time - start_train_time

                    # print(
                    #     "\r\033[KEPOCH: "
                    #     + str(epoch + 1)
                    #     + "/"
                    #     + str(self.epochs)
                    #     + "|||"
                    #     + "TRAIN: "
                    #     + str(batch_index + 1)
                    #     + "/"
                    #     + str(len(self.train_loader))
                    #     + "|||"
                    #     + "TIME: "
                    #     + str(int(train_time))
                    #     + "s"
                    #     + "|||"
                    #     + "COST: "
                    #     + str(train_cost_fn.item()),
                    #     end="",
                    # )

                model.eval()
                with no_grad():
                    for batch_index, (inputs, labels) in enumerate(
                        self.test_loader
                    ):
                        # print('TEST BATCH: ', batch_index + 1)
                        output = model(inputs)

                        # Compute cost function
                        test_cost_fn = self.loss_fn(
                            output.float(), labels.float()
                        )

                        # Compute correct predictions
                        _, predicted_labels = torch_max(output, 1)
                        true_labels = argmax(labels, dim=1)
                        correct_predictions: Tensor = (
                            predicted_labels == true_labels
                        ).sum()
                        test_accuracy: Tensor = correct_predictions / inputs.size(
                            0
                        )

                        # Add metrics to lists
                        epoch_test_costs.append(test_cost_fn)
                        epoch_test_accuracies.append(test_accuracy)


                        # print(
                        #     "\r\033[KEPOCH: "
                        #     + str(epoch + 1)
                        #     + "/"
                        #     + str(self.epochs)
                        #     + "|||"
                        #     + "TEST: "
                        #     + str(batch_index + 1)
                        #     + "/"
                        #     + str(len(self.test_loader))
                        #     + "|||"
                        #     + "ACCURACY: "
                        #     + str(test_accuracy.item()),
                        #     end="",
                        # )

                # Compute epoch averages for graphical representation
                avg_epoch_train_cost = sum(epoch_train_costs) / len(epoch_train_costs)
                avg_epoch_train_accuracy = sum(epoch_train_accuracies) / len(
                    epoch_train_accuracies
                )
                avg_epoch_test_cost = sum(epoch_test_costs) / len(
                    epoch_test_costs
                )
                avg_epoch_test_accuracy = sum(epoch_test_accuracies) / len(
                    epoch_test_accuracies
                )

                # Record the model's parameters
                results.models.append(model.state_dict())
                
                if (
                    type(avg_epoch_train_cost) == Tensor
                    and type(avg_epoch_train_accuracy) == Tensor
                    and type(avg_epoch_test_cost) == Tensor
                    and type(avg_epoch_test_accuracy) == Tensor
                ):
                    
                    # Record training metrics
                    results.avg_epoch_train_costs.append(avg_epoch_train_cost.detach())
                    results.avg_epoch_train_accuracies.append(
                        avg_epoch_train_accuracy.detach()
                    )
                    results.avg_epoch_test_costs.append(
                        avg_epoch_test_cost.detach()
                    )
                    results.avg_epoch_test_accuracies.append(
                        avg_epoch_test_accuracy.detach()
                    )

                    # Update csv file
                    csvwriter.writerow(
                        [
                            epoch,
                            avg_epoch_train_cost.item(),
                            avg_epoch_train_accuracy.item(),
                            avg_epoch_test_cost.item(),
                            avg_epoch_test_accuracy.item(),
                        ]
                    )
                    end_epoch_time = time.time()
                    epoch_time = end_epoch_time - start_epoch_time

                    print(
                        "EPOCH: "
                        + str(epoch + 1)
                        + "/"
                        + str(self.epochs)
                        + "|||"
                        + "TIME: "
                        + str(int(epoch_time))
                        + "s"
                        + "|||"
                        + "TRAIN COST: "
                        + str(round(avg_epoch_train_cost.item(),2))
                        + "|||"
                        + "TRAIN ACCURACY: "
                        + str(round(avg_epoch_train_accuracy.item(),2))
                        + "|||"
                        + "TEST COST: "
                        + str(round(avg_epoch_test_cost.item(),2))
                        + "|||"
                        + "TEST ACCURACY: "
                        + str(round(avg_epoch_test_accuracy.item(),2)),
                    )
        return results

plot.py

In [ ]:
def plot_results(results: TrainingResult):
    """Plot the results of training and test, saving each plot individually as a PDF
    into a specified folder.

    Arguments:
    ----------
    results : TrainingResult
        The results to be plotted.
    output_folder : str, optional
        The name of the folder where the plots will be saved.
        Defaults to "training_plots".
    """
    output_folder = results.plot_path

    # Create the output folder if it doesn't exist
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    avg_epoch_train_costs = results.avg_epoch_train_costs
    avg_epoch_train_accuracies = results.avg_epoch_train_accuracies
    avg_epoch_test_costs = results.avg_epoch_test_costs
    avg_epoch_test_accuracies = results.avg_epoch_test_accuracies

    # Plot and save Train Cost as PDF
    plt.figure(figsize=(6, 4))
    plt.plot(avg_epoch_train_costs, label="Train cost function")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Cost on the training set")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, "train_cost.pdf")) # Save in the specified folder
    plt.close()

    # Plot and save Test Cost as PDF
    plt.figure(figsize=(6, 4))
    plt.plot(avg_epoch_test_costs, label="Test cost function")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Cost on the test set")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, "test_cost.pdf")) # Save in the specified folder
    plt.close()

    # Plot and save Train Accuracy as PDF
    plt.figure(figsize=(6, 4))
    plt.plot(avg_epoch_train_accuracies, label="Train accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Accuracy on the training set")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, "train_accuracy.pdf")) # Save in the specified folder
    plt.close()

    # Plot and save Test Accuracy as PDF
    plt.figure(figsize=(6, 4))
    plt.plot(avg_epoch_test_accuracies, label="Test accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Accuracy on the test set")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, "test_accuracy.pdf")) # Save in the specified folder
    plt.close()

main.py

In [ ]:
# Define configuration
dataset_folder_path = 'Tetris'
BATCH_SIZE = 40
noise = 'depolarizing'
NOISE_PROB = 0.01
KERNEL_SIZE = 2
device = qml.device("default.qubit")
feature_map = 'zz'
ansatz = 'real_amplitudes'
FEATURE_MAP_REPS = 1
ANSATZ_REPS = 1
CLASSES = num_classes(dataset_folder_path=dataset_folder_path)
show_circuit = config["show_circuit"]
loss_fn = MSELoss()
EPOCHS = 100
LEARNING_RATE = 0.01

# Set random seed for reproducibility
manual_seed(42)

# Load data
train_loader, test_loader, _ = load_dataset(
    dataset_folder_path=dataset_folder_path,
    batch_size=BATCH_SIZE
)

# Create device
num_qubits : int = int(KERNEL_SIZE * KERNEL_SIZE)
wires : List = list(range(num_qubits))
# device = qml.device("default.mixed", wires=wires)
device : qml.devices
if isinstance(NOISE_PROB, float):
    if NOISE_PROB > 1 or NOISE_PROB < 0:
        raise ValueError("NOISE_PROB must be in the range [0, 1]")
    elif NOISE_PROB > 0:
        device = qml.device("default.mixed", wires=wires)
    elif NOISE_PROB == 0:
        device = qml.device("default.qubit", wires=wires)

# Create the cnn
model = create_cnn(
    train_loader = test_loader,
    dataset_folder_path = dataset_folder_path,
    kernel_size = KERNEL_SIZE,
    device = device,
    noise = noise,
    noise_prob = NOISE_PROB,
    feature_map = feature_map,
    ansatz = ansatz,
    feature_map_reps = FEATURE_MAP_REPS,
    ansatz_reps = ANSATZ_REPS,
    classes = CLASSES,
    show_circuit = show_circuit,
)

# Perform training and validation of the model
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE
)

# Get results
results = trainer.train_and_validate()

# Plot results
if results is not None:
    plot_results(results)